# Categorizing Transactions Model

### Setup

In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

from imblearn.over_sampling import RandomOverSampler
from xgboost import XGBClassifier
from google.oauth2.service_account import Credentials
import gspread


### Cleaning and Parse Amount and Dates

In [12]:
scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

creds = Credentials.from_service_account_file("transactions-project-483516-f68f182fae44.json", scopes=scopes)
client = gspread.authorize(creds)

sheet = client.open_by_key("1UoIaxcnFor39N_KcIwnq8JWaauPzrDWEv_S5PFcca1M").worksheet("All Transactions")
rows = sheet.get_all_records()
df = pd.DataFrame(rows)
df = df[['Account', 'Amount', 'Category', 'Date', 'Description']]

# make sure categories are valid
df['Category'] = df['Category']
df = df[df['Category'].notna()]

# Clean Amounts like '-$5.70', '$3.90', '$1,234.56'
df['Amount'] = (
    df['Amount']
    .astype(str)
    .str.replace('[\$,]', '', regex=True)  # remove $ and commas
    .astype(float)
)

# Parse Date
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date', 'Amount'])  # drop rows with invalid date/amount


<>:22: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<>:22: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
C:\Users\hokie\AppData\Local\Temp\ipykernel_3956\3123096516.py:22: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
  .str.replace('[\$,]', '', regex=True)  # remove $ and commas


### 3. Derived Features

In [13]:
# Numeric & Date features
df['abs_amount'] = df['Amount'].abs()
df['is_income'] = (df['Amount'] > 0).astype(int)
df['weekday'] = df['Date'].dt.weekday
df['day'] = df['Date'].dt.day
df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)
df['Month'] = df["Month"] = df["Date"].dt.month_name()

### Prepare features and labels

In [14]:
feature_cols = ['Description', 'Account', 'Month', 'abs_amount', 'is_income', 'weekday', 'is_weekend']
X = df[feature_cols]
y = df['Category']

In [15]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

### Train / Test Split

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

### Column Transformer for mixed features

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=10000), 'Description'),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['Account', 'Month']),
        ('num', 'passthrough', ['abs_amount', 'is_income', 'weekday', 'is_weekend'])
    ]
)

### Handle Class imbalance

In [18]:
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_train, y_train)

### Build the pipeline with XGBoost

In [19]:
model = XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    use_label_encoder=False,
    n_jobs=-1,
    random_state=42
)

pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('classifier', model)
])


### Train the model

In [20]:
pipeline.fit(X_resampled, y_resampled)


c:\Users\hokie\OneDrive\Desktop\git\budget_sheet_scripting\.venv\sheets_venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [14:44:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('text', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transf

### Evaluate Performance

In [22]:
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00         6
           1       0.67      1.00      0.80         6
           2       0.50      1.00      0.67         1
           3       1.00      1.00      1.00         1
           5       0.60      1.00      0.75         3
           6       1.00      0.67      0.80         3
           7       0.85      0.93      0.89       186
           8       1.00      1.00      1.00         6
           9       1.00      1.00      1.00         6
          10       0.67      0.57      0.62         7
          11       0.81      0.51      0.63        43
          12       1.00      1.00      1.00         2
          13       1.00      1.00      1.00         6
          14       1.00      1.00      1.00         6
          15       1.00      0.33      0.50         3
          16       1.00      1.00      1.00         6
          17       0.94      0.73      0.82        22
          18       0.40    

c:\Users\hokie\OneDrive\Desktop\git\budget_sheet_scripting\.venv\sheets_venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hokie\OneDrive\Desktop\git\budget_sheet_scripting\.venv\sheets_venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hokie\OneDrive\Desktop\git\budget_sheet_scripting\.venv\sheets_venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `

### Save the model

In [23]:
import joblib

joblib.dump(pipeline, 'transaction_model.pkl')
joblib.dump(le, 'label_encoder.pkl')

['label_encoder.pkl']

### Predict for missing categories

In [46]:
import gspread
from google.oauth2.service_account import Credentials
import pandas as pd
import joblib
import numpy as np

# --------------------------
# Load trained model + label encoder
# --------------------------
pipeline = joblib.load("transaction_model.pkl")  # contains {'pipeline': pipeline, 'label_encoder': le}
le = joblib.load('label_encoder.pkl')

# --------------------------
# Google Sheets auth
# --------------------------
scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

creds = Credentials.from_service_account_file("transactions-project-483516-1dcb6dfc2520.json", scopes=scopes)
client = gspread.authorize(creds)

sheet = client.open_by_key("1UoIaxcnFor39N_KcIwnq8JWaauPzrDWEv_S5PFcca1M").worksheet("All Transactions")
rows = sheet.get_all_records()
df = pd.DataFrame(rows)

# Amount
df['Amount'] = df['Amount'].astype(str).str.replace('[\$,]', '', regex=True).astype(float)

# Date
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date', 'Amount'])

# Derived features
df['abs_amount'] = df['Amount'].abs()
df['is_income'] = (df['Amount'] > 0).astype(int)
df['weekday'] = df['Date'].dt.weekday
df['day'] = df['Date'].dt.day
df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)

# --------------------------
# Preprocess sheet data
# --------------------------
df["Category"] = df["Category"].fillna("").str.strip()
to_predict = df[df["Category"] == ""].copy()
print(f"Predicting {len(to_predict)} transactions")

# --------------------------
# Prepare model input
# --------------------------
X_pred = to_predict[['Description', 'Account', 'Month', 'abs_amount', 'is_income', 'weekday', 'is_weekend']]

# --------------------------
# Predict categories + probabilities
# --------------------------
probs = pipeline.predict_proba(X_pred)
predicted_indices = np.argmax(probs, axis=1)
predicted_categories = le.inverse_transform(predicted_indices)
confidences = probs[np.arange(len(probs)), predicted_indices]

# Pretty-print categories
def prettify_category(cat: str) -> str:
    return cat.replace("_", " ").title()



# --------------------------
# Apply confidence threshold
# --------------------------
CONF_THRESHOLD = 0.8
final_categories = [
    cat if conf >= CONF_THRESHOLD else "" 
    for cat, conf in zip(predicted_categories, confidences)
]
final_confidences = [
    conf if conf >= CONF_THRESHOLD else np.nan
    for conf in confidences
]

# --------------------------
# Update DataFrame
# --------------------------
df.loc[to_predict.index, "Category"] = final_categories
df.loc[to_predict.index, "Confidence"] = final_confidences

# --------------------------
# Push updates to Google Sheets
# --------------------------
headers = sheet.row_values(1)
category_col = headers.index("Category") + 1

# Add confidence column if not exist
if "Confidence" in headers:
    confidence_col = headers.index("Confidence") + 1
else:
    sheet.update_cell(1, len(headers)+1, "Confidence")
    confidence_col = len(headers) + 1

# Convert confidence to standard Python float (or None for NaN)
final_confidences = [
    float(conf) if not pd.isna(conf) else None
    for conf in final_confidences
]

# Now push updates
updates = []
for idx in to_predict.index:
    sheet_row = idx + 2  # df index → sheet row
    updates.append({
        "range": gspread.utils.rowcol_to_a1(sheet_row, category_col),
        "values": [[df.loc[idx, "Category"]]]
    })
    updates.append({
        "range": gspread.utils.rowcol_to_a1(sheet_row, confidence_col),
        "values": [[final_confidences[to_predict.index.get_loc(idx)]]]
    })

if updates:
    sheet.batch_update(updates)
    print(f"Updated {len(to_predict)} transactions with confidence threshold of {CONF_THRESHOLD*100:.0f}%")

Predicting 5 transactions
Updated 5 transactions with confidence threshold of 80%


### Process all statements

In [10]:
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials
import numpy as np
import re

# --------------------------
# Config
# --------------------------
SPREADSHEET_KEY = "1UoIaxcnFor39N_KcIwnq8JWaauPzrDWEv_S5PFcca1M"
SHEET_NAME = "All Transactions"

# --------------------------
# Auth
# --------------------------
scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

creds = Credentials.from_service_account_file(
    "transactions-project-483516-1dcb6dfc2520.json",
    scopes=scopes
)

client = gspread.authorize(creds)
sheet = client.open_by_key(SPREADSHEET_KEY).worksheet(SHEET_NAME)

# --------------------------
# Read sheet
# --------------------------
rows = sheet.get_all_records()
df = pd.DataFrame(rows)

print(f"Loaded {len(df)} rows")

# --------------------------
# Clean Amount
# --------------------------
def clean_amount(val):
    if val is None:
        return ""

    s = str(val).strip()

    if s == "":
        return ""

    # Remove leading apostrophe from Sheets "'-$5.00"
    s = s.lstrip("'")

    # Detect sign
    sign = -1 if "-" in s else 1

    # Remove everything except digits + dot
    num = re.sub(r"[^\d.]", "", s)

    if num == "":
        return ""

    return round(sign * float(num), 2)

df["Amount"] = df["Amount"].apply(clean_amount)

# --------------------------
# Push cleaned column back
# --------------------------
amount_col_idx = sheet.row_values(1).index("Amount") + 1

# Convert to column vector
amount_values = [[v] for v in df["Amount"].tolist()]

# Update entire Amount column (starting from row 2)
sheet.update(
    f"{chr(64 + amount_col_idx)}2:{chr(64 + amount_col_idx)}{len(amount_values) + 1}",
    amount_values,
    value_input_option="USER_ENTERED"
)

print("✅ Amount column cleaned and converted to numeric floats.")


Loaded 1979 rows
✅ Amount column cleaned and converted to numeric floats.


C:\Users\Jake Huffman\AppData\Local\Temp\ipykernel_15944\2258582847.py:74: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  sheet.update(
